# Week 56, Day 1: Logistic regression: probabilities, cross-entropy and boundaries

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w056/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Gradient descent on the cross-entropy

Write cross_entropy(w) and grad(w) for the design A (a column of ones and two inputs). Run 2,000 steps of gradient descent with step size 0.002 and record the loss every step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(561)
n = 400
Xin = r.normal(size=(n, 2))
A = np.c_[np.ones(n), Xin]
w_true = np.array([-0.5, 2.0, -1.0])
y = (r.random(n) < 1 / (1 + np.exp(-A @ w_true))).astype(float)

def cross_entropy(w):
    p = 1 / (1 + np.exp(-A @ w))
    return ...

def grad(w):
    p = 1 / (1 + np.exp(-A @ w))
    return ...

w_gd = np.zeros(3); losses = []
for step in range(2000):
    w_gd -= 0.002 * grad(w_gd)
    losses.append(cross_entropy(w_gd))
print("true weights:  ", w_true); print("gradient descent:", np.round(w_gd, 3))
print(f"loss after 1, 100 and 2000 steps: {losses[0]:.2f}, {losses[99]:.2f}, {losses[-1]:.2f}")
plt.semilogy(np.array(losses) - min(losses) + 1e-3); plt.xlabel("step"); plt.ylabel("loss above its minimum"); plt.show()

In [ ]:
import numpy as np
def test_loss():
    "cross_entropy matches the formula"
    w0 = np.array([0.1, -0.2, 0.3]); p = 1 / (1 + np.exp(-A @ w0))
    assert abs(cross_entropy(w0) + np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))) < 1e-9
def test_grad():
    "grad matches a finite-difference derivative"
    w0 = np.array([0.1, -0.2, 0.3]); e = np.array([0, 1e-6, 0])
    assert abs(grad(w0)[1] - (cross_entropy(w0 + e) - cross_entropy(w0 - e)) / 2e-6) < 1e-3
def test_fit():
    "gradient descent reduces the loss and recovers the weights roughly"
    assert losses[-1] < losses[0] and np.abs(w_gd - w_true).max() < 0.5

test_loss()
test_grad()
test_fit()
print("All checks passed")

### Exercise: Converging in a few steps

Write newton(A, y, steps), returning the weights and the loss after each step. Compare the number of steps it needs with gradient descent.

In [ ]:
def newton(A, y, steps=12):
    w = np.zeros(A.shape[1]); ls = []
    for _ in range(steps):
        p = 1 / (1 + np.exp(-A @ w))
        H = ...
        w = w - np.linalg.solve(H, A.T @ (p - y))
        ls.append(cross_entropy(w))
    return w, ls

w_nt, nt_losses = newton(A, y)
print("Newton weights:", np.round(w_nt, 4))
print("loss after each step:", np.round(nt_losses[:6], 4))

In [ ]:
import numpy as np
def test_newton():
    "newton returns the same weights as long gradient descent, within 0.01"
    assert np.abs(w_nt - w_gd).max() < 0.01
def test_fast():
    "the loss stops changing (to 1e-8) within 10 Newton steps"
    assert np.argmax(np.abs(np.diff(nt_losses)) < 1e-8) < 10

test_newton()
test_fast()
print("All checks passed")

### Exercise: Weights that run away

Run gradient descent without a penalty on the 40-trial, 50-neuron decoding problem and record the weight norm. Then fit with an L2 penalty (add lam * w to the gradient, intercept excluded) and compare the norms and the test accuracies on 2,000 new trials.

In [ ]:
g = np.random.default_rng(562)
Nn, n_tr = 50, 40
mu = g.normal(size=Nn) * 0.3                     # mean response difference between the two stimuli
def trials(n):
    lab = g.integers(0, 2, n).astype(float)
    return np.c_[np.ones(n), (2 * lab[:, None] - 1) * mu / 2 + g.normal(size=(n, Nn))], lab
A_tr, y_tr = trials(n_tr); A_te, y_te = trials(2000)

def fit_gd(A, y, lam=0.0, steps=20000, lr=0.05):
    w = np.zeros(A.shape[1]); norms = []
    for s in range(steps):
        p = 1 / (1 + np.exp(-A @ w))
        gpen = lam * np.r_[0, w[1:]]
        w -= lr * (A.T @ (p - y) / len(y) + gpen)
        if s % 1000 == 0: norms.append(np.linalg.norm(w[1:]))
    return w, norms

acc = lambda A, y, w: np.mean((A @ w > 0) == (y == 1))
w_free, norms = fit_gd(A_tr, y_tr)
acc_train, acc_test_free = acc(A_tr, y_tr, w_free), acc(A_te, y_te, w_free)
w_pen, _ = ...
norm_pen, acc_test_pen = np.linalg.norm(w_pen[1:]), acc(A_te, y_te, w_pen)
print(f"no penalty: training accuracy {acc_train:.2f}, weight norm after 5k, 10k, 20k steps: {norms[5]:.1f}, {norms[10]:.1f}, {norms[-1]:.1f}; test accuracy {acc_test_free:.3f}")
print(f"penalty 0.1: weight norm {norm_pen:.2f}; test accuracy {acc_test_pen:.3f}")

In [ ]:
import numpy as np
def test_separable():
    "the unpenalised training accuracy reaches 100% while the weight norm keeps growing"
    assert acc_train == 1.0 and norms[-1] > norms[10] > norms[5]
def test_penalty():
    "with a penalty the norm stays bounded and the test accuracy is at least as good"
    assert norm_pen < 0.5 * norms[-1] and acc_test_pen >= acc_test_free - 0.02

test_separable()
test_penalty()
print("All checks passed")